In [ ]:
# ================================================================
# PLACEMENTPREDICT - STUDENT CLUSTERING
# K-Means + Hierarchical Clustering + DBSCAN
# ================================================================

# -----------------------------
# 0. IMPORT LIBRARIES
# -----------------------------
import os
import glob
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage


# -----------------------------
# SETTINGS
# -----------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("=" * 70)
print("PLACEMENTPREDICT STUDENT CLUSTERING")
print("K-Means / Hierarchical / DBSCAN")
print("=" * 70)


# ================================================================
# 1. UPLOAD CSV
# ================================================================

print("\nChecking for CSV files...")

csv_files = glob.glob("*.csv")

if len(csv_files) == 0:
    print("No CSV found. Please upload your dataset.")
    uploaded = files.upload()

    csv_files = glob.glob("*.csv")

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV file found. Please upload your placement dataset."
    )

print("\nCSV files found:")
for i, file in enumerate(csv_files):
    print(f"{i + 1}. {file}")


# If there is only one CSV, automatically use it
if len(csv_files) == 1:
    file_name = csv_files[0]

else:
    # Try to automatically find placement dataset
    placement_files = [
        f for f in csv_files
        if "placement" in f.lower()
    ]

    if len(placement_files) >= 1:
        file_name = placement_files[0]
    else:
        file_name = csv_files[0]

print(f"\nUsing dataset: {file_name}")


# ================================================================
# 2. LOAD DATA
# ================================================================

df = pd.read_csv(file_name)

print("\nDataset loaded successfully!")
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

print("\nColumns in dataset:")
print(df.columns.tolist())


# ================================================================
# 3. DEFINE FEATURES
# ================================================================

NUMERIC_COLS = [
    "CGPA",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore"
]

CATEGORICAL_COLS = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",
    "ExtraCurricular"
]

TARGET_COL = "PlacementStatus"


# ================================================================
# 4. CHECK REQUIRED COLUMNS
# ================================================================

required_columns = (
    NUMERIC_COLS +
    CATEGORICAL_COLS +
    [TARGET_COL]
)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if len(missing_columns) > 0:

    print("\n❌ Missing columns:")
    for col in missing_columns:
        print(" -", col)

    print("\nYour dataset does not have the exact column names expected")
    print("by this program.")

    raise ValueError(
        "Missing required columns. Check the column names above."
    )

print("\n✅ All required columns are present.")


# ================================================================
# 5. CLEAN NUMERIC DATA
# ================================================================

print("\nCleaning numeric features...")

for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# Median imputation
imputer = SimpleImputer(strategy="median")

num_df = pd.DataFrame(
    imputer.fit_transform(df[NUMERIC_COLS]),
    columns=NUMERIC_COLS,
    index=df.index
)


# ================================================================
# 6. CLEAN CATEGORICAL DATA
# ================================================================

print("Encoding categorical features...")

cat_data = df[CATEGORICAL_COLS].copy()

# Convert missing categorical values to "Unknown"
for col in CATEGORICAL_COLS:
    cat_data[col] = cat_data[col].fillna("Unknown").astype(str)

# One-hot encoding
cat_df = pd.get_dummies(
    cat_data,
    drop_first=True,
    dtype=float
)


# ================================================================
# 7. COMBINE FEATURES
# ================================================================

feature_df = pd.concat(
    [num_df, cat_df],
    axis=1
)

print(f"\nTotal clustering features: {feature_df.shape[1]}")


# ================================================================
# 8. STANDARDIZE FEATURES
# ================================================================

scaler = StandardScaler()

X = scaler.fit_transform(feature_df)

print(f"Final feature matrix shape: {X.shape}")

print("\nIMPORTANT:")
print("PlacementStatus is NOT used as a clustering feature.")


# ================================================================
# 9. CHOOSE K USING ELBOW + SILHOUETTE
# ================================================================

print("\n" + "=" * 70)
print("K-MEANS: FINDING BEST K")
print("=" * 70)

# Never request more samples than available
sil_sample_size = min(5000, len(X))

rng = np.random.RandomState(RANDOM_STATE)

sil_sample_idx = rng.choice(
    len(X),
    sil_sample_size,
    replace=False
)

X_sil_sample = X[sil_sample_idx]


# Use reasonable k values
max_k = min(8, len(X) - 1)

if max_k < 2:
    raise ValueError("Dataset is too small for clustering.")

k_range = range(2, max_k + 1)

inertias = []
sil_scores = []

for k in k_range:

    print(f"Testing k = {k}...")

    km = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=RANDOM_STATE
    )

    labels = km.fit_predict(X)

    inertia = km.inertia_

    # Silhouette on sample
    sample_labels = km.predict(X_sil_sample)

    # Silhouette requires at least 2 clusters
    if len(np.unique(sample_labels)) >= 2:
        sil = silhouette_score(
            X_sil_sample,
            sample_labels
        )
    else:
        sil = -1

    inertias.append(inertia)
    sil_scores.append(sil)


# ================================================================
# 10. ELBOW + SILHOUETTE PLOT
# ================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4.5)
)


# Elbow
axes[0].plot(
    list(k_range),
    inertias,
    marker="o"
)

axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].grid(True, alpha=0.3)


# Silhouette
axes[1].plot(
    list(k_range),
    sil_scores,
    marker="o"
)

axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].grid(True, alpha=0.3)


plt.tight_layout()

plt.savefig(
    "kmeans_elbow_silhouette.png",
    dpi=150
)

plt.close(fig)

print("\nSaved:")
print("kmeans_elbow_silhouette.png")


# ================================================================
# 11. SELECT BEST K
# ================================================================

best_k = list(k_range)[
    int(np.argmax(sil_scores))
]

print("\nSilhouette scores:")

for k, score in zip(k_range, sil_scores):
    print(
        f"k = {k}  ->  silhouette = {score:.4f}"
    )

print(f"\nSelected K = {best_k}")


# ================================================================
# 12. FINAL K-MEANS
# ================================================================

print("\n" + "=" * 70)
print("FINAL K-MEANS")
print("=" * 70)

kmeans = KMeans(
    n_clusters=best_k,
    n_init=10,
    random_state=RANDOM_STATE
)

km_labels = kmeans.fit_predict(X)


# Calculate silhouette
if len(np.unique(km_labels)) >= 2:

    km_silhouette = silhouette_score(
        X_sil_sample,
        km_labels[sil_sample_idx]
    )

else:
    km_silhouette = -1


print(
    f"K-Means silhouette score: {km_silhouette:.4f}"
)

cluster_sizes = (
    pd.Series(km_labels)
    .value_counts()
    .sort_index()
)

print("\nCluster sizes:")

for cluster, size in cluster_sizes.items():
    print(
        f"Cluster {cluster}: {size:,} students"
    )


# ================================================================
# 13. PLACEMENT STATUS CLEANING
# ================================================================

print("\nProcessing PlacementStatus...")

placement = df[TARGET_COL].copy()


# Convert common text labels to 0/1
placement_clean = (
    placement
    .astype(str)
    .str.strip()
    .str.lower()
)

placement_mapping = {
    "yes": 1,
    "no": 0,
    "placed": 1,
    "not placed": 0,
    "true": 1,
    "false": 0,
    "1": 1,
    "0": 0
}

placement_numeric = placement_clean.map(
    placement_mapping
)

# If already numeric
numeric_original = pd.to_numeric(
    placement,
    errors="coerce"
)

placement_numeric = placement_numeric.fillna(
    numeric_original
)


# ================================================================
# 14. CLUSTER PROFILES
# ================================================================

profiled = num_df.copy()

profiled["KMeansCluster"] = km_labels
profiled["PlacementStatus"] = placement_numeric.values


summary = (
    profiled
    .groupby("KMeansCluster")
    .agg(
        n_students=("PlacementStatus", "size"),

        placement_rate=(
            "PlacementStatus",
            "mean"
        ),

        avg_CGPA=("CGPA", "mean"),

        avg_AttendancePercent=(
            "AttendancePercent",
            "mean"
        ),

        avg_AptitudeTestScore=(
            "AptitudeTestScore",
            "mean"
        ),

        avg_SoftSkillsRating=(
            "SoftSkillsRating",
            "mean"
        ),

        avg_CodingTestScore=(
            "CodingTestScore",
            "mean"
        ),

        avg_MockInterviewScore=(
            "MockInterviewScore",
            "mean"
        ),

        avg_Internships=(
            "Internships",
            "mean"
        ),

        avg_Projects=(
            "Projects",
            "mean"
        )
    )
    .round(3)
)


# Convert placement rate to percentage
summary["placement_rate_percent"] = (
    summary["placement_rate"] * 100
).round(2)


print("\n" + "=" * 70)
print("CLUSTER PROFILES")
print("=" * 70)

print(
    summary.to_string()
)


# Save profile
summary.to_csv(
    "cluster_profiles.csv"
)

print("\nSaved:")
print("cluster_profiles.csv")


# ================================================================
# 15. HIERARCHICAL CLUSTERING
# ================================================================

print("\n" + "=" * 70)
print("HIERARCHICAL CLUSTERING")
print("=" * 70)

# Maximum 5000 because hierarchical clustering can become
# computationally expensive for large datasets.

subsample_size = min(5000, len(X))

sub_idx = rng.choice(
    len(X),
    subsample_size,
    replace=False
)

X_sub = X[sub_idx]


hc = AgglomerativeClustering(
    n_clusters=best_k,
    linkage="ward"
)

hc_labels = hc.fit_predict(X_sub)


if len(np.unique(hc_labels)) >= 2:

    sil_hc = silhouette_score(
        X_sub,
        hc_labels
    )

else:
    sil_hc = -1


print(
    f"Hierarchical silhouette score: {sil_hc:.4f}"
)


# ================================================================
# 16. DBSCAN
# ================================================================

print("\n" + "=" * 70)
print("DBSCAN")
print("=" * 70)

db = DBSCAN(
    eps=4.0,
    min_samples=15
)

db_labels = db.fit_predict(X_sub)


unique_db = set(db_labels)

n_db_clusters = len(
    unique_db - {-1}
)

n_noise = int(
    np.sum(db_labels == -1)
)


print(
    f"DBSCAN clusters: {n_db_clusters}"
)

print(
    f"Noise points: {n_noise}"
)


# DBSCAN silhouette only if valid
non_noise_mask = db_labels != -1

if (
    n_db_clusters >= 2
    and np.sum(non_noise_mask) > n_db_clusters
):

    db_silhouette = silhouette_score(
        X_sub[non_noise_mask],
        db_labels[non_noise_mask]
    )

    print(
        f"DBSCAN silhouette: {db_silhouette:.4f}"
    )

else:

    db_silhouette = None

    print(
        "DBSCAN silhouette cannot be calculated "
        "because there are not enough valid clusters."
    )


# ================================================================
# 17. PCA
# ================================================================

print("\n" + "=" * 70)
print("PCA VISUALIZATION")
print("=" * 70)

pca = PCA(
    n_components=2,
    random_state=RANDOM_STATE
)

X_pca = pca.fit_transform(X)

coords_sub = X_pca[sub_idx]


# ================================================================
# 18. K-MEANS PCA PLOT
# ================================================================

fig, ax = plt.subplots(
    figsize=(7, 5.5)
)

scatter = ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=km_labels,
    cmap="tab10",
    s=8,
    alpha=0.6
)

ax.set_title(
    f"K-Means - {best_k} Clusters"
)

ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")

plt.tight_layout()

plt.savefig(
    "kmeans_pca.png",
    dpi=150
)

plt.close(fig)

print("Saved:")
print("kmeans_pca.png")


# ================================================================
# 19. HIERARCHICAL + DBSCAN PCA
# ================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5.2)
)


# Hierarchical
axes[0].scatter(
    coords_sub[:, 0],
    coords_sub[:, 1],
    c=hc_labels,
    cmap="tab10",
    s=8,
    alpha=0.6
)

axes[0].set_title(
    f"Hierarchical - {best_k} Clusters"
)

axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")


# DBSCAN
noise_mask = db_labels == -1

if np.any(~noise_mask):

    axes[1].scatter(
        coords_sub[~noise_mask, 0],
        coords_sub[~noise_mask, 1],
        c=db_labels[~noise_mask],
        cmap="tab10",
        s=8,
        alpha=0.6
    )


if np.any(noise_mask):

    axes[1].scatter(
        coords_sub[noise_mask, 0],
        coords_sub[noise_mask, 1],
        s=8,
        alpha=0.5,
        marker="x",
        label="Noise"
    )

    axes[1].legend(
        loc="upper right",
        fontsize=8
    )


axes[1].set_title(
    f"DBSCAN - {n_db_clusters} Clusters"
)

axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")


plt.tight_layout()

plt.savefig(
    "hierarchical_dbscan_pca.png",
    dpi=150
)

plt.close(fig)

print("Saved:")
print("hierarchical_dbscan_pca.png")


# ================================================================
# 20. DENDROGRAM
# ================================================================

print("\nCreating dendrogram...")

dendro_size = min(
    80,
    len(X_sub)
)

dendro_idx = rng.choice(
    len(X_sub),
    dendro_size,
    replace=False
)

Z = linkage(
    X_sub[dendro_idx],
    method="ward"
)


plt.figure(
    figsize=(12, 5)
)

dendrogram(Z)

plt.title(
    "Hierarchical Clustering Dendrogram"
)

plt.xlabel("Student Index")
plt.ylabel("Distance")

plt.tight_layout()

plt.savefig(
    "dendrogram.png",
    dpi=150
)

plt.close()

print("Saved:")
print("dendrogram.png")


# ================================================================
# 21. SAVE FINAL DATASET
# ================================================================

df_out = df.copy()

df_out["KMeansCluster"] = km_labels

df_out.to_csv(
    "placement_predict_with_clusters.csv",
    index=False
)

print("\nSaved:")
print("placement_predict_with_clusters.csv")


# ================================================================
# 22. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("🎉 CLUSTERING COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"\nStudents processed : {len(df):,}")
print(f"Features used      : {feature_df.shape[1]}")
print(f"Selected K         : {best_k}")
print(f"K-Means silhouette  : {km_silhouette:.4f}")
print(f"Hierarchical score : {sil_hc:.4f}")

if db_silhouette is not None:
    print(
        f"DBSCAN silhouette   : {db_silhouette:.4f}"
    )
else:
    print(
        "DBSCAN silhouette   : Not available"
    )

print("\nOutput files:")

output_files = [
    "kmeans_elbow_silhouette.png",
    "kmeans_pca.png",
    "hierarchical_dbscan_pca.png",
    "dendrogram.png",
    "cluster_profiles.csv",
    "placement_predict_with_clusters.csv"
]

for f in output_files:
    if os.path.exists(f):
        print("✅", f)
    else:
        print("❌", f)

print("\nDone! 🎉")

PLACEMENTPREDICT STUDENT CLUSTERING
K-Means / Hierarchical / DBSCAN

Checking for CSV files...

CSV files found:
1. placement_predict_50k_adjusted.csv

Using dataset: placement_predict_50k_adjusted.csv

Dataset loaded successfully!
Rows    : 50,000
Columns : 21

Columns in dataset:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'PlacementStatus', 'IsAnomaly']

✅ All required columns are present.

Cleaning numeric features...
Encoding categorical features...

Total clustering features: 33
Final feature matrix shape: (50000, 33)

IMPORTANT:
PlacementStatus is NOT used as a clustering feature.

K-MEANS: FINDING BEST K
Testing k = 2...
Testing k = 3...
Testing k = 4...
Testing k = 5...
Testing k = 6...
Testing k = 7...
Testing k = 8...

Saved:
